# ProteinGym

In [2]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/grid/koo/home/schilder/projects/GenomeEncoder/')
import pandas as pd
import sys
sys.path.append("code")
import src.haplosaurus as hs
import src.ESM as ESM
import src.utils as utils
import src.proteingym as pg

## Download data

In [2]:
pg_resources = pg.download_resources(error=False)

Error downloading zero_shot_clinical_substitutions_scores.zip: 404 Client Error: Not Found for url: https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.1/zero_shot_clinical_substitutions_scores.zip


In [4]:
{k:f"{len(v)} file(s)" for k,v in pg_resources.items()}

{'DMS_ProteinGym_substitutions': '217 file(s)',
 'DMS_ProteinGym_indels': '66 file(s)',
 'zero_shot_substitutions_scores': '10201 file(s)',
 'zero_shot_indels_scores': '1541 file(s)',
 'DMS_supervised_substitutions_scores': '1 file(s)',
 'DMS_supervised_indels_scores': '1 file(s)',
 'DMS_msa_files': '194 file(s)',
 'DMS_msa_weights': '402 file(s)',
 'ProteinGym_AF2_structures': '197 file(s)',
 'clinical_ProteinGym_substitutions': '2525 file(s)',
 'clinical_ProteinGym_indels': '1555 file(s)',
 'clinical_msa_files': '4073 file(s)',
 'clinical_msa_weights': '2525 file(s)',
 'zero_shot_clinical_indels_scores': '31093 file(s)',
 'cv_folds_singles_substitutions': '217 file(s)',
 'cv_folds_multiples_substitutions': '69 file(s)',
 'cv_folds_indels': '66 file(s)'}

In [5]:
# Read and concatenate all clinical substitution files
def concat_csvs(pg_resources,
                fkey='clinical_ProteinGym_substitutions'):
    from tqdm.auto import tqdm
    clinical_subs_dfs = []
    for csv_path in tqdm(pg_resources[fkey],
                        desc="Reading files in " + fkey):
        df = pd.read_csv(csv_path, index_col=0)
        # Add source file name as column
        df['source_file'] = os.path.basename(csv_path)
        clinical_subs_dfs.append(df)
    return pd.concat(clinical_subs_dfs, ignore_index=True)

### Substitutions

In [6]:
clinical_subs_df = concat_csvs(pg_resources, fkey='clinical_ProteinGym_substitutions')

Reading files in clinical_ProteinGym_substitutions:   0%|          | 0/2525 [00:00<?, ?it/s]

In [21]:
# Report stats
haplotypes_per_protein = 37.5
print(clinical_subs_df['protein'].nunique()*haplotypes_per_protein,
      "haplotypes across",len(clinical_subs_df), 
      "substitutions in",clinical_subs_df['protein'].nunique(),"proteins")

sum(clinical_subs_df.groupby('mutant').size()*haplotypes_per_protein)

94687.5 haplotypes across 62727 substitutions in 2525 proteins


2352262.5

In [ ]:
# Using all benign/pathogenic variants per protein
print(sum(clinical_subs_df.groupby('protein').size()*haplotypes_per_protein))
# Using only 1 benign/pathogenic variant per protein
print(clinical_subs_df['protein'].nunique()*2*haplotypes_per_protein)

2352262.5
189375.0


### Indels

In [ ]:
clinical_indels_df = concat_csvs(pg_resources, fkey='clinical_ProteinGym_indels')

Reading files in clinical_ProteinGym_indels:   0%|          | 0/1555 [00:00<?, ?it/s]

In [14]:
# Report stats
haplotypes_per_protein = 37.5
print(clinical_indels_df['refseq_unique_id'].nunique()*haplotypes_per_protein,
      "estimated haplotypes across",len(clinical_indels_df), 
      "substitutions in",clinical_indels_df['refseq_unique_id'].nunique(),"proteins")
print("Total estimated haplotype sequences:",
      clinical_indels_df['refseq_unique_id'].nunique()*haplotypes_per_protein)

32175.0 haplotype sequences across 2878 substitutions in 858 proteins


In [ ]:
# Using all benign/pathogenic variants per protein
print(sum(clinical_indels_df.groupby('refseq_unique_id').size()*haplotypes_per_protein))
# Using only 1 benign/pathogenic variant per protein
print(clinical_indels_df['refseq_unique_id'].nunique()*2*haplotypes_per_protein)

76462.5
64350.0
